# CS781 Phase 4: reviewed runner, recovered own course data

Attach both private datasets: original model inputs and recovered course data plus runner. Select GPU T4 x2. Six actual adaptations run two at a time, with exact checkpoint resume, train-only model selection and verified prediction ZIP exports. Data were recovered from own saved course cache using the existing official dev/test split IDs; original JSON formatting, duplicate copies and expert fields are absent. Test labels are absent.

Run All for a GPU preflight and actual progress check. Save & Run All can run a background Kaggle version; avoid a simultaneous interactive job. Per-session service limits still apply. All experiments are truthfully after the original deadline. Output is under /kaggle/working/CS781_Phase4_EvidenceV2_20261022.


In [ ]:
import os,time,sys,subprocess,importlib.util
from pathlib import Path
SESSION_STARTED=time.time()
os.environ['PHASE4_SESSION_END']=str(SESSION_STARTED+11.4*3600)
os.environ['OMP_NUM_THREADS']='2';os.environ['MKL_NUM_THREADS']='2'
os.environ['TOKENIZERS_PARALLELISM']='false'
needed={'ijson':'ijson>=3.2,<4','sklearn':'scikit-learn>=1.4,<2','tokenizers':'tokenizers'}
missing=[package for module,package in needed.items() if importlib.util.find_spec(module) is None]
if missing:subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('Dependencies ready; notebook session clock started.')


In [ ]:
import hashlib,io,json,zipfile,shutil
EXPECTED_SOURCE={'phase3_rr/__init__.py': 'f38a2af63c8fe34b0e5fe200c4a9c71fb161c9ad6a37bf46ec7307eb24d00ae6', 'phase3_rr/archive_inputs.py': '900109016e17f4170100f04a630b7259e41eee51442d04d015b40a85d0512391', 'phase3_rr/common.py': 'f808cde35564328d8f377dabb79da21e8ef1001623d9897d7b789fd2251e6535', 'phase3_rr/data.py': '00084b779497b1480c9574583faef1b6700a7bd92c4a1ff2fe5c1d1cd0ffc02a', 'phase3_rr/models.py': '14fc38d447e1fcb8d663d00932b24344c74d7c383cf7b543a8b90cc4e97c4eea', 'phase3_rr/train.py': 'db1532039e21f531b81f15b568a7ab97ce86779df5df4b5b0ff1f4299fbd4f63', 'phase3_rr/vendor/__init__.py': '4e813963bf49a55dffeba982788a12a8d854c7e7a4e6ed19b300b4ea35cef565', 'phase3_rr/vendor/backbones/cs781/__init__.py': '4a3613ae45d376f074874f3924962127b635a12fc6a03c5cf2c2d2e9cc3c1c49', 'phase3_rr/vendor/backbones/cs781/checkpoints.py': 'f70275cc12a1ca3bda33c4103dc2f697b0a374f8dcd5a3f30ab9f9f3f3b08418', 'phase3_rr/vendor/backbones/cs781/cli.py': 'e6695a4ec962094bf8fba96d61e61182f421440af61475644ae5dfe9223bfd8f', 'phase3_rr/vendor/backbones/cs781/cli_b.py': '471b80b2b4485a7bbc13a196fbfa452c7faa536fce85c7f9293742012248aa33', 'phase3_rr/vendor/backbones/cs781/cli_c.py': '0e6b8d4aa1b942aeb214881d1d9074e524ce90dfed641b5438a41601874c3aa6', 'phase3_rr/vendor/backbones/cs781/contracts.py': 'b4d17bf29617b79bff3c3507cf01245dbe16ad550c2c394a15aaddedd0733689', 'phase3_rr/vendor/backbones/cs781/corpus.py': '5706941f90284ba4481524a6ffe93c9f108bfcadcb0845107a107c57891098bf', 'phase3_rr/vendor/backbones/cs781/evaluate.py': '9ca3eb30f79592adfce063cdb47a34cb21a84b33b7406c8e888ce980267dba2a', 'phase3_rr/vendor/backbones/cs781/examples.py': '372e9d40984f66bda6873dbd87382416544eee410984651c27a7d000c38b1eba', 'phase3_rr/vendor/backbones/cs781/examples_alt.py': '7260c5b156dcd046c23545ab851c25014aef76a81ca73b612216de80a4c83052', 'phase3_rr/vendor/backbones/cs781/generate.py': '9a3bc31bd2a5e0e7fdb342e81e0fc20c031beab13711dc3bbf96b5728dcc8d03', 'phase3_rr/vendor/backbones/cs781/generate_b.py': 'e92ed53fcf2f6853eda5cc8a48bdacac0c2b92478e4677d4161dc3b4f6eba6ba', 'phase3_rr/vendor/backbones/cs781/generate_c.py': '43be46a4714d76a835f83a792b72210ceb80b889ce30675e13555243bc7d598e', 'phase3_rr/vendor/backbones/cs781/io.py': '6f7c8510cff6eee2d6cc82dbb77cd0d3d3cd7063670c0da37363ca7d600f45e4', 'phase3_rr/vendor/backbones/cs781/models/__init__.py': '0da8971507f5266ea3b85759054329394979d08d7f8e4896472fd471f78d3d49', 'phase3_rr/vendor/backbones/cs781/models/a.py': 'b5eb47496093388acf0798fa44fe2155116bd30a3fd1a9fbec3f6a71b5dca268', 'phase3_rr/vendor/backbones/cs781/models/b.py': '9919cc6a3ac87fb080acf6ac2be2df84bfb0d1d6d314e53fcf6c977bde14c057', 'phase3_rr/vendor/backbones/cs781/models/c.py': 'd9c16e1b53c23db7fd2a2be3e83f09c30642012fd31a63dc8282b7e39dc98147', 'phase3_rr/vendor/backbones/cs781/own_bpe.py': '2cac66fc09ee09136174f3837676a7578c71fa7d59a23bada83a18203029e07e', 'phase3_rr/vendor/backbones/cs781/prediction_progress.py': '3d35956bdecc1f88890010be3cd15527bd33ce463ff48b2a304db470f0a505c4', 'phase3_rr/vendor/backbones/cs781/prepare.py': 'efb7368de000b0e9f44595072b0f7f146e1809ef6f34e799859b14b33de79f9e', 'phase3_rr/vendor/backbones/cs781/splits.py': '2434ecd26e762919b66d4586ab530615c578d677a385bb1c78eb06f4d2cfee83', 'phase3_rr/vendor/backbones/cs781/submission.py': 'c36de151bccce0f844827c33087ec9c1b3163cb2ed7a32e45d7bf198cad1063d', 'phase3_rr/vendor/backbones/cs781/tokenizer.py': 'e838c764e9dc2848c1ea04f9f51b261aab1afe5607cfe8cf7322ab5984555558', 'phase3_rr/vendor/backbones/cs781/train.py': 'a2367df0b50efb36660b067940356b953340240483feb787d2108ee4258d6588', 'phase3_rr/vendor/backbones/cs781/train_b.py': '3d0bac658a06799ece3b90ca23a53f6a2627749188f79418ef093344bdde0758', 'phase3_rr/vendor/backbones/cs781/train_c.py': '63e88403d9e6fbdd754b635d08f9e38d4b1898ff1373e927723e15e8274d0c4e', 'phase3_rr/vendor/backbones/legalspan/__init__.py': 'c24781d92781386918ca9056ca807b62493db4b5096d078dfcb0b1809b48b37e', 'phase3_rr/vendor/backbones/legalspan/bpe.py': '2cac66fc09ee09136174f3837676a7578c71fa7d59a23bada83a18203029e07e', 'phase3_rr/vendor/backbones/legalspan/data.py': '48b7af4ae83e715210f201777ae4c876876af550a9d1f09b485d29250ba7a06b', 'phase3_rr/vendor/backbones/legalspan/generate.py': '315c7067041e2314df5375a0d0b29899af0972366a21da0a546b43687d2748bd', 'phase3_rr/vendor/backbones/legalspan/model.py': '0423c932eaf22a919139ed433f1a5aee5e8a2a3720edf4e3b9d19ac5023435d0', 'phase3_rr/vendor/backbones/legalspan/pipeline.py': 'c20587337d8ac74b24ec517020d9752ea508f31e4b802e5219080f9728692cbb', 'phase3_rr/vendor/backbones/legalspan/train.py': '86a6d99aba0a153d0b0759662ba4257cd3e7955c0283a2550747f943b29017a2', 'phase3_rr/vendor/evaluate_phase3.py': '5b2babf1cb42a6f211c3264b7dc4be61566cd22df9eb2aae8b609ea0ee2f0b1f', 'phase3_rr/vendor/legal_d/__init__.py': 'fe77b5788e408a4d7c774eccd3cd8b359527cc580ce82eacc38fd5294e6c8989', 'phase3_rr/vendor/legal_d/model.py': '35c3ebc178de3a5797884542bb739d225a3ea7fd32af9774b759187c73c272fc', 'phase3_rr/vendor/SOURCES.md': 'b3cacaa420e9604392614d9e6628cc591b9d9c9f494b97b765b16d61fa12b3f4', 'phase3_rr/vendor/UnicodeData-13.0.0.txt': 'bdbffbbfc8ad4d3a6d01b5891510458f3d36f7170422af4ea2bed3211a73e8bb', 'phase3_rr/vendor/UnicodeData-14.0.0.txt': '36018e68657fdcb3485f636630ffe8c8532e01c977703d2803f5b89d6c5feafb', 'phase4_cjpe/__init__.py': 'f18c25fd56e59b628ab217396751039d3abbbf12ef6b0a935337a9c7b58c8576', 'phase4_cjpe/common.py': '0546d963cdf031283a87ce032076ffc832b7f49f15f6af1b0a9d4fc21589e069', 'phase4_cjpe/data.py': '95e6823bd24a6172ef3d7840112046fc818c7b069fa557638bca61adf9577703', 'phase4_cjpe/evaluate_phase4.py': '16f71e82d66b8b7a9e5289db22f3285edea99a4e6df8e9617b5b9e78699e3766', 'phase4_cjpe/export.py': '4a743510077a1fccd518774df56ecade58a5bcdac39fe69fd533408ad1f45090', 'phase4_cjpe/kaggle_runner.py': '8a8a4f82e083f080b93641eec1381b43720cb8e85582bb2a608e58aaf1eaed77', 'phase4_cjpe/models.py': '2a6fc936dd37531472824414c64bc0e561275f90c9448708e08625cbf46b9562', 'phase4_cjpe/online_teacher.py': 'b4ae71b323ced68019127241e991d87c1a6c0995d773f33a6efdd3c914e0aed4', 'phase4_cjpe/runtime.py': '066e8144daf6f5a1976cd52da1a0892955f6a7460e92a41480b28c65920b4469', 'phase4_cjpe/train.py': '2224e8e18e00f77ec52bda3b6c85a1de3428ec2236582791ca995137333248b9'}
manifests=[]
for p in Path('/kaggle/input').rglob('SOURCE_MANIFEST.json'):
    try:
        if json.loads(p.read_text())==EXPECTED_SOURCE:manifests.append(p)
    except (ValueError,OSError):pass
assert len(manifests)==1, 'Attach the private CS781 recovered data and runner dataset.'
CODE_ROOT=Path('/kaggle/working/phase4_evidence_v2_source')
CODE_ROOT.mkdir(parents=True,exist_ok=True)
stream=io.BytesIO()
with zipfile.ZipFile(stream,'w',zipfile.ZIP_DEFLATED) as archive:
    for relative,digest in EXPECTED_SOURCE.items():
        contents=(manifests[0].parent/relative).read_bytes()
        assert hashlib.sha256(contents).hexdigest()==digest, 'Source hash mismatch: '+relative
        dest=CODE_ROOT/relative;dest.parent.mkdir(parents=True,exist_ok=True);dest.write_bytes(contents)
        archive.writestr(relative,contents)
raw=stream.getvalue()
sys.path.insert(0,str(CODE_ROOT))
PERSIST=Path('/kaggle/working/CS781_Phase4_EvidenceV2_20261022')
WORK=Path('/kaggle/temp/cs781_phase4_evidence_v2')
PERSIST.mkdir(parents=True,exist_ok=True);WORK.mkdir(parents=True,exist_ok=True)
(PERSIST/'EXACT_SOURCE.zip').write_bytes(raw)
from phase4_cjpe.common import sha,atomic_json
EXPECTED={'a0': {'checkpoint.pt': '1e8aca9800b87e952ed82e54307dd874220845d6a8545d2568bb743993cfb9dc', 'tokenizer.json': '9b32abd6b09b9c30fcd6cccac236d5815a8740f35d4b70993cd4390a18a72e0c'}, 'b1': {'checkpoint.pt': 'd93a2f1796e90b9fdbe4a7b542f76eb6065dab619feed4a1ef5a9952e2376a1f', 'tokenizer.json': '5c0bd30290e0bd6919253b1ff864f52053924808aabd33482df92cc4bd471c8e'}, 'c1': {'checkpoint.pt': '1903b848b39ca9f335c8d9e3add532d5f49718d539b130e6a8348d98d802433e', 'tokenizer.json': '5c0bd30290e0bd6919253b1ff864f52053924808aabd33482df92cc4bd471c8e'}, 'd222': {'checkpoint.pt': '08a737d002a71a90ad21f350e25a76f1e70eb172badf0671ff8818d96ea5c46f', 'tokenizer.json': '5c0bd30290e0bd6919253b1ff864f52053924808aabd33482df92cc4bd471c8e'}}
BUNDLES={}
for manifest_path in Path('/kaggle/input').rglob('INPUT_MANIFEST.json'):
    try:manifest=json.loads(manifest_path.read_text())
    except (ValueError,OSError):continue
    family=manifest.get('family')
    if family not in EXPECTED:continue
    if family in BUNDLES:raise ValueError('Ambiguous duplicate family input: '+family)
    bundle=manifest_path.parent
    assert manifest['sha256']==EXPECTED[family], 'Wrong original model manifest'
    for filename,expected_sha in EXPECTED[family].items():
        assert sha(bundle/filename)==expected_sha, 'Input member hash mismatch: '+str(bundle/filename)
    BUNDLES[family]=bundle
    print('Verified original model bundle:',family,bundle)
assert set(BUNDLES)==set(EXPECTED), 'Attach all four own model bundles using Add Input.'
assert shutil.disk_usage(PERSIST).free>12*1024**3, 'Need at least 12 GiB free output space for selected and resumable states.'
atomic_json(PERSIST/'INPUT_PROVENANCE.json',{f:{'path':str(p),'sha256':EXPECTED[f]} for f,p in BUNDLES.items()})
print('Four own input bundles verified. No checkpoint has been trained or changed.')


In [ ]:
# Exact recovered data from own saved course cache and official split exports.
EXPECTED_RECOVERED={'Train/train.json': {'sha256': '49bfc5f4d441833e2468e4c280c390ce361e2bfaffd5fbf1ad54157cddda164c', 'bytes': 611284455}, 'Train/dev.json': {'sha256': 'a071b080d3ffcd9789009cf46f6e4f1008142de0f44db9b97df9b64b6ee201c9', 'bytes': 22387465}, 'Test/test.json': {'sha256': '88913c10c288a6f77a030d0391904d40b645fc242f2774f0ef8216e9f5afa792', 'bytes': 35050736}}
matches=[]
for p in Path('/kaggle/input').rglob('RECOVERY_MANIFEST.json'):
    try:
        if json.loads(p.read_text()).get('files')==EXPECTED_RECOVERED:matches.append(p)
    except (ValueError,OSError):pass
assert len(matches)==1, 'Attach the private recovered course data dataset.'
recovery=json.loads(matches[0].read_text())
DATA_ROOT=WORK/'official_data'
for relative,metadata in EXPECTED_RECOVERED.items():
    source=matches[0].parent/relative
    assert source.stat().st_size==metadata['bytes'] and sha(source)==metadata['sha256'], 'Data hash mismatch: '+relative
    dest=DATA_ROOT/relative;dest.parent.mkdir(parents=True,exist_ok=True)
    if not dest.exists():dest.symlink_to(source)
atomic_json(PERSIST/'OFFICIAL_DATA_PROVENANCE.json',recovery)
from phase4_cjpe.data import index_data
db,DATA_AUDIT=index_data(DATA_ROOT,WORK/'shared_official_index',progress_root=PERSIST/'data_preflight')
assert {k:v['cases'] for k,v in DATA_AUDIT['counts'].items()}=={'train':32305,'dev':994,'test':1517}
atomic_json(PERSIST/'DATA_AUDIT.json',DATA_AUDIT)
print('Recovered data passed exact hashes and cross-split checks:',DATA_AUDIT['counts'])


In [ ]:
# Actual GPU / own-checkpoint preflight. Synthetic token IDs; no optimizer step.
import gc,torch
from phase4_cjpe.models import make_model
assert torch.cuda.is_available() and torch.cuda.device_count()==2, 'Choose GPU T4 x2 before launch.'
print('Allocated:',[(i,torch.cuda.get_device_name(i)) for i in range(2)])
SMOKE=[]
for n,(family,method) in enumerate([('d222','lorma'),('b1','partial'),('c1','full'),('d222','adapters'),('a0','full'),('c1','bitfit')]):
    index=n%2;device=torch.device('cuda:'+str(index));torch.cuda.set_device(index)
    model,evidence=make_model(BUNDLES[family],family,method,8,16);model.to(device).train()
    with torch.autocast(device_type='cuda',dtype=torch.float16):
        logits=model([[1,4,5,6,7,8,9,2],[1,9,8,7,6,5,4,2]],[0.,1.],device)
        loss=torch.nn.functional.cross_entropy(logits.float(),torch.tensor([1],device=device))
    loss.backward()
    gradients=[p.grad for p in model.backbone.parameters() if p.requires_grad and p.grad is not None]
    assert gradients and any(bool(p.abs().sum()>0) for p in gradients), 'No adaptation gradient'
    assert torch.isfinite(logits).all() and all(bool(torch.isfinite(g).all()) for g in gradients), 'Nonfinite preflight'
    SMOKE.append({'family':family,'method':method,'gpu':index,'parameters':evidence['parameters'],
      'trainable':evidence['trainable_parameters'],'finite':True,'nonzero_backbone_gradient':True})
    del gradients,model,logits,loss;gc.collect();torch.cuda.empty_cache()
atomic_json(PERSIST/'GPU_PREFLIGHT.json',SMOKE)
print('All six own-checkpoint adaptation preflights passed:',SMOKE)


In [ ]:
# Six new, truthfully after-deadline experiments. Keep configs unchanged for exact resume.
definitions=[('01_D222_LoRMA','d222','lorma',2e-4,2.5),('02_B1_Partial','b1','partial',2e-5,2.5),
 ('03_C1_Full','c1','full',1e-5,3.5),('05_D222_Adapters','d222','adapters',3e-4,2.5),
 ('04_A0_Distillation','a0','full',1e-5,2.5),('06_C1_BitFit','c1','bitfit',2e-4,2.5)]
JOBS=[]
for name,family,method,lr,hours in definitions:
    name=name+'_KaggleEvidenceV2'
    cfg=dict(name=name,family=family,method=method,experiment='distillation' if family=='a0' else 'adaptation',
       rank=8,alpha=16,seed=781,max_tokens=256,chunks_per_case=8,region_chars=8192,
       max_train_cases=12000,holdout_per_class=256,max_epochs=4,accumulation=8,
       backbone_lr=lr,head_lr=3e-4,training_hours=hours,eval_every=100,save_every=10,patience=5,
       teacher_hours=1.5,kd_temperature=2.,kd_weight=.5,stall_minutes=45,export_reserve_minutes=45,
       deadline_utc='2026-10-21T18:29:00Z',allow_after_deadline=True,
       run_root=str(PERSIST/'runs'/name),work_root=str(WORK),data_root=str(DATA_ROOT),bundle=str(BUNDLES[family]))
    if family=='a0':cfg['teacher_bundle']=str(BUNDLES['d222'])
    JOBS.append(cfg)
atomic_json(PERSIST/'RUN_SCHEDULE.json',JOBS)
import sqlite3
with sqlite3.connect(db) as conn:EXPECTED_TEST_IDS=[r[0] for r in conn.execute('SELECT id FROM cases WHERE split="test" ORDER BY id')]
print('Schedule:',[(j['name'],j['training_hours']) for j in JOBS])
print('Session remaining hours:',round((float(os.environ['PHASE4_SESSION_END'])-time.time())/3600,2))


In [ ]:
# Launch two isolated workers and collect each verified ZIP as it appears.
from phase4_cjpe.kaggle_runner import run_schedule
from IPython.display import display,FileLink
def completed(record):
    if record.get('verified_submission'):
        display(FileLink(record['submission']))
    elif record.get('state'):
        print('EXPERIMENT TERMINAL:',record['name'],record['state'])
RESULTS=run_schedule(JOBS,PERSIST,EXPECTED_TEST_IDS,on_worker_complete=completed)
display(FileLink(str(PERSIST/'ALL_VERIFIED_SUBMISSIONS.zip')))
print('Completed/paused/failed records are in RUN_SUMMARY.json. Raw checkpoints are under runs/.')
